In [ ]:
pip install youtube-transcript-api

In [ ]:
import re
import youtube_transcript_api
from youtube_transcript_api import YouTubeTranscriptApi

In [ ]:
def get_youtube_id(url):
    # Regex pattern to match standard, shortened, embedded, and shorts URLs
    pattern = r'(?:https?:\/\/)?(?:www\.)?(?:youtube\.com\/(?:[^\/\n\s]+\/\S+\/|(?:v|e(?:mbed)?|shorts)\/|\S*?[?&]v=)|youtu\.be\/)([a-zA-Z0-9_-]{11})'

    match = re.search(pattern, url)
    if match:
        return match.group(1)
    return "Invalid YouTube URL"

# Test cases
video_id = get_youtube_id("https://youtu.be/xDQL3vWwcp0?si=xQJ2aKy4Dhsij2Nh")

ytt_api = YouTubeTranscriptApi()
fetched_transcript = ytt_api.fetch(video_id)

In [ ]:
fetched_transcript

In [ ]:
fetched_transcript_joined = " ".join([line.text for line in fetched_transcript])

# this is better bcz it holds the transcript in the memory which is the good habit
# for later to be used by the LLM for summerizing and all the operations wee want to do laterr on.

In [ ]:
fetched_transcript_joined

In [ ]:
pip install deepmultilingualpunctuation

In [ ]:
from deepmultilingualpunctuation import PunctuationModel
from transformers import pipeline

punc_pipe = pipeline(
    "token-classification",
    model="oliverguhr/fullstop-punctuation-multilang-large",
    device="cpu"
)

print(punc_pipe(fetched_transcript_joined))


In [ ]:
transcript_list = ytt_api.list(video_id)

print(
    fetched_transcript.video_id,
    fetched_transcript.language,
    fetched_transcript.language_code,
    # whether it has been manually created or generated by YouTube
    fetched_transcript.is_generated,
    # whether this transcript can be translated or not

)

In [ ]:
import openai

In [ ]:
prompt=f'"Summarize this text: \ntext ="{fetched_transcript_joined}"'

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

# 4. Make the call using the free model
chat_completion = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    max_tokens=500
)

# 5. Print the output
print(chat_completion.choices[0].message.content)


In [ ]:
chat_completion